In [1]:
import wmfdata as wmf
import pandas as pd
import numpy as np
from wmfdata import spark,hive
import plotly.express as px

In [6]:
# Query

regional_uniques = wmf.spark.run("""
SELECT 
    cmd.wmf_region,
    ud.country_code,
    cmd.canonical_country_name AS country, -- Use canonical country name from gdi
    AVG(CASE WHEN ud.month IN (7, 8, 9) THEN ud.uniques_estimate ELSE NULL END) AS previous_metric_avg,
    AVG(CASE WHEN ud.month IN (10, 11, 12) THEN ud.uniques_estimate ELSE NULL END) AS current_metric_avg
FROM 
    wmf.unique_devices_per_project_family_monthly ud
JOIN 
    gdi.country_meta_data cmd ON ud.country_code = cmd.country_code_iso_2
WHERE 
    ud.project_family = 'wikipedia'
    AND ud.year = 2023
    AND ud.month IN (7, 8, 9, 10, 11, 12)
GROUP BY 
    cmd.wmf_region,
    ud.country_code,
    cmd.canonical_country_name -- Group by canonical country name

""")




In [15]:
df = regional_uniques.copy()
df['absolute_change'] = df['current_metric_avg'] - df['previous_metric_avg']

# Calculate Regional and Global Totals for the delta
regional_delta_total = df.groupby('wmf_region')['absolute_change'].sum().reset_index(name='total_region_delta')
global_delta_total = df['absolute_change'].sum()

# Merge the total regional delta back to the original DataFrame
df = pd.merge(df, regional_delta_total, on='wmf_region')

# Calculating Proportions for current metrics towards the regional and global totals
df['proportion_current_regional'] = (df['current_metric_avg'] / df.groupby('wmf_region')['current_metric_avg'].transform('sum')) * 100
df['proportion_current_global'] = (df['current_metric_avg'] / df['current_metric_avg'].sum()) * 100

# Update decomposition formula calculations to focus on country_code
df['formula_result_regional'] = abs((df['absolute_change'] * df['proportion_current_regional']) +
                                    ((df['proportion_current_regional'] - df.groupby('country_code')['proportion_current_regional'].transform('mean')) * (df['previous_metric_avg'] - df.groupby('wmf_region')['previous_metric_avg'].transform('sum'))))

df['formula_result_global'] = abs((df['absolute_change'] * df['proportion_current_global']) +
                                  ((df['proportion_current_global'] - df['proportion_current_global'].mean()) * (df['previous_metric_avg'] - df['previous_metric_avg'].sum())))

# Calculate the Total Sum of 'formula_result' per Region and Globally
df['formula_result_regional_sum'] = df.groupby('wmf_region')['formula_result_regional'].transform('sum')
df['total_formula_result_global'] = df['formula_result_global'].sum()

# Calculate percentages for formula results
df['formula_result_percentage_regional'] = (df['formula_result_regional'] / df['formula_result_regional_sum']) * 100
df['formula_result_percentage_global'] = (df['formula_result_global'] / df['total_formula_result_global']) * 100

# Prepare the final DataFrame for output with requested columns
df['month'] = 'Quarter' 

final_df = df[['month', 'wmf_region', 'country', 'current_metric_avg', 'previous_metric_avg', 'absolute_change', 
               'proportion_current_regional', 'formula_result_percentage_regional', 'proportion_current_global', 'formula_result_percentage_global']]

# Save the sorted DataFrame to a CSV file
print("Saving CSV file")
final_df.to_csv("unique_devices_time_series_analysis_quarterly.csv", index=False)

# Display the DataFrame
final_df

Saving CSV file


,month,wmf_region,country,current_metric_avg,previous_metric_avg,absolute_change,proportion_current_regional,formula_result_percentage_regional,proportion_current_global,formula_result_percentage_global
0,Quarter,Northern & Western Europe,Jersey,7.510967e+04,7.601200e+04,-9.023333e+02,0.018911,0.000003,0.004520,0.294562
1,Quarter,Northern & Western Europe,Luxembourg,8.461593e+05,7.003270e+05,1.458323e+05,0.213047,0.006001,0.050924,0.260316
2,Quarter,Northern & Western Europe,Åland,2.285800e+04,2.438800e+04,-1.530000e+03,0.005755,0.000002,0.001376,0.296885
3,Quarter,Northern & Western Europe,Faroe Islands,3.533133e+04,3.620733e+04,-8.760000e+02,0.008896,0.000002,0.002126,0.296331
4,Quarter,Northern & Western Europe,Isle of Man,7.124033e+04,7.337600e+04,-2.135667e+03,0.017937,0.000007,0.004287,0.294734
...,...,...,...,...,...,...,...,...,...,...
242,Quarter,South Asia,Nepal,1.918250e+06,2.063692e+06,-1.454413e+05,1.166321,0.062344,0.115444,0.212660
243,Quarter,South Asia,Maldives,1.782933e+05,1.382957e+05,3.999767e+04,0.108405,0.001594,0.010730,0.289981
244,Quarter,South Asia,Sri Lanka,2.293245e+06,2.254674e+06,3.857133e+04,1.394323,0.019766,0.138012,0.196063
245,Quarter,South Asia,India,1.427275e+08,1.396513e+08,3.076186e+06,86.780149,98.112304,8.589625,5.458610


## Mobile Wiki breakdown

In [8]:
# Query

regional_uniques_mobile = wmf.spark.run("""
SELECT 
    cmd.wmf_region,
    ud.country_code,
    cmd.canonical_country_name AS country, -- Use canonical country name from gdi
    AVG(CASE WHEN ud.month IN (7, 8, 9) THEN ud.uniques_estimate ELSE NULL END) AS previous_metric,
    AVG(CASE WHEN ud.month IN (10, 11, 12) THEN ud.uniques_estimate ELSE NULL END) AS current_metric
FROM 
    wmf.unique_devices_per_domain_monthly ud
JOIN 
    gdi.country_meta_data cmd ON ud.country_code = cmd.country_code_iso_2
WHERE 
    ud.domain LIKE '%.m.wikipedia%'
    AND ud.year = 2023
GROUP BY 
    cmd.wmf_region,
    ud.country_code,
    cmd.canonical_country_name -- Group by canonical country name


""")


In [17]:

df = regional_uniques_mobile.copy()

# Calculate Absolute Change directly
df['absolute_change'] = df['current_metric'] - df['previous_metric']

# Calculate Regional and Global Totals
regional_totals = df.groupby('wmf_region')['current_metric', 'previous_metric'].sum().reset_index()
global_current_metric_total = df['current_metric'].sum()
global_previous_metric_total = df['previous_metric'].sum()

# Add Global Totals to df for proportion calculations
df['global_current_metric_total'] = global_current_metric_total
df['global_previous_metric_total'] = global_previous_metric_total

# Calculating Proportions and Formula Results
df['proportion_current_regional'] = df['current_metric'] / df.groupby('wmf_region')['current_metric'].transform('sum')
df['proportion_previous_regional'] = df['previous_metric'] / df.groupby('wmf_region')['previous_metric'].transform('sum')
df['proportion_current_global'] = df['current_metric'] / global_current_metric_total * 100
df['proportion_previous_global'] = df['previous_metric'] / global_previous_metric_total * 100

df['formula_result_regional'] = abs((df['absolute_change'] * df['proportion_current_regional']) +
                                    ((df['proportion_current_regional'] - df['proportion_previous_regional']) * (df['previous_metric'] - df.groupby('wmf_region')['previous_metric'].transform('sum'))))
df['formula_result_global'] = abs((df['absolute_change'] * df['proportion_current_global']) +
                                  ((df['proportion_current_global'] - df['proportion_previous_global']) * (df['previous_metric'] - global_previous_metric_total)))

# Calculate the Total Sum of 'formula_result' per Region and Globally
df['formula_result_regional_sum'] = df.groupby('wmf_region')['formula_result_regional'].transform('sum')
df['total_formula_result_global'] = df['formula_result_global'].sum()

# Calculate percentages
df['formula_result_percentage_regional'] = (df['formula_result_regional'] / df['formula_result_regional_sum']) * 100
df['formula_result_percentage_global'] = (df['formula_result_global'] / df['total_formula_result_global']) * 100

# Add ranking logic
df['rank_simple_calculation'] = df.groupby('wmf_region')['absolute_change'].rank(method='dense', ascending=False)
df['rank_formula_result'] = df.groupby('wmf_region')['formula_result_percentage_regional'].rank(method='dense', ascending=False)






df['month'] = 'Quarter'
final_df = df[['month', 'wmf_region', 'country', 'current_metric', 'previous_metric', 'absolute_change', 
               'proportion_current_regional', 'formula_result_percentage_regional', 'proportion_current_global', 'formula_result_percentage_global']]

# Sort the DataFrame by 'wmf_region', and 'absolute_change'
final_df = df.sort_values(by=['wmf_region', 'absolute_change'], ascending=[True, False])

# Saving the DataFrame to a CSV file
print("Saving CSV file")
final_df.to_csv("unique_devices_time_series_analysis_mobile_quarterly.csv", index=False, header = False)

# Display the DataFrame
final_df

/tmp/ipykernel_15263/970794784.py:7: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  regional_totals = df.groupby('wmf_region')['current_metric', 'previous_metric'].sum().reset_index()


Saving CSV file


,wmf_region,country_code,country,previous_metric,current_metric,absolute_change,global_current_metric_total,global_previous_metric_total,proportion_current_regional,proportion_previous_regional,...,proportion_previous_global,formula_result_regional,formula_result_global,formula_result_regional_sum,total_formula_result_global,formula_result_percentage_regional,formula_result_percentage_global,rank_simple_calculation,rank_formula_result,month
193,Central & Eastern Europe & Central Asia,RU,Russia,131777.449161,148108.334311,16330.885150,4.514823e+06,4.561661e+06,0.247516,0.245637,...,2.888804,3281.859989,1.681552e+06,35273.970965,3.250093e+07,9.303914,5.173858,1.0,5.0,Quarter
84,Central & Eastern Europe & Central Asia,TR,Turkey,65017.699308,72934.114594,7916.415286,4.514823e+06,4.561661e+06,0.121886,0.121195,...,1.425308,639.055443,8.421554e+05,35273.970965,3.250093e+07,1.811691,2.591173,2.0,15.0,Quarter
216,Central & Eastern Europe & Central Asia,KZ,Kazakhstan,17242.536706,24773.588813,7531.052106,4.514823e+06,4.561661e+06,0.041401,0.032141,...,0.377988,4496.551533,7.717299e+05,35273.970965,3.250093e+07,12.747506,2.374486,3.0,1.0,Quarter
213,Central & Eastern Europe & Central Asia,UA,Ukraine,49263.266535,56021.974585,6758.708050,4.514823e+06,4.561661e+06,0.093623,0.091828,...,1.079941,241.595570,7.176766e+05,35273.970965,3.250093e+07,0.684912,2.208173,4.0,24.0,Quarter
124,Central & Eastern Europe & Central Asia,UZ,Uzbekistan,8658.315519,13724.106557,5065.791038,4.514823e+06,4.561661e+06,0.022935,0.016139,...,0.189806,3470.890510,5.182885e+05,35273.970965,3.250093e+07,9.839807,1.594688,5.0,4.0,Quarter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,Sub-Saharan Africa,MZ,Mozambique,2244.308348,2054.989547,-189.318800,4.514823e+06,4.561661e+06,0.018558,0.021181,...,0.049199,268.514011,1.678309e+04,8891.791143,3.250093e+07,3.019797,0.051639,51.0,11.0,Quarter
233,Sub-Saharan Africa,ZM,Zambia,2299.493243,2093.376563,-206.116681,4.514823e+06,4.561661e+06,0.018905,0.021702,...,0.050409,286.038441,1.842108e+04,8891.791143,3.250093e+07,3.216882,0.056679,52.0,9.0,Quarter
91,Sub-Saharan Africa,MU,Mauritius,1828.389098,1611.750419,-216.638679,4.514823e+06,4.561661e+06,0.014555,0.017256,...,0.040082,278.034237,1.997604e+04,8891.791143,3.250093e+07,3.126864,0.061463,53.0,10.0,Quarter
79,Sub-Saharan Africa,ZA,South Africa,20197.337688,18196.873031,-2000.464657,4.514823e+06,4.561661e+06,0.164333,0.190617,...,0.442763,1925.367678,1.795600e+05,8891.791143,3.250093e+07,21.653316,0.552477,54.0,1.0,Quarter
